In [1]:
# imports for EDA 
import pandas as pd
import numpy as  np
import matplotlib.pyplot as plt
import seaborn as sns
import re


In [2]:
df = pd.read_csv('train.csv')
df.head()

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0


In [3]:
df.describe()

,toxic,severe_toxic,obscene,threat,insult,identity_hate
count,159571.000000,159571.000000,159571.000000,159571.000000,159571.000000,159571.000000
mean,0.095844,0.009996,0.052948,0.002996,0.049364,0.008805
std,0.294379,0.099477,0.223931,0.054650,0.216627,0.093420
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


## Regex preprocessing

In [4]:
import pandas as pd
import numpy as np
import re
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

label_cols = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

# STEP 1: Text Cleaning Function
def clean_text(text):
    if not isinstance(text, str):
        return ""
    
    text = text.lower()
    
    # 1. Replace line breaks and tabs with a single space
    text = re.sub(r'[\r\n\t]+', ' ', text)
    
    # 2. Remove IP addresses (common in Wikipedia edits)
    text = re.sub(r'\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}', '', text)
    
    # 3. Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    
    # 4. Standardize contraction apostrophes
    text = re.sub(r"n't", " not", text)
    text = re.sub(r"'s", " is", text)
    text = re.sub(r"'re", " are", text)
    text = re.sub(r"'ve", " have", text)
    text = re.sub(r"'ll", " will", text)
    text = re.sub(r"'d", " would", text)
    
    # 5. Keep only alphanumeric characters and basic punctuation
    text = re.sub(r'[^a-zA-Z0-9\s!\?]', '', text)
    
    # 6. Collapse multiple whitespaces into a single space
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

print("Cleaning comment_text column...")
df['clean_text'] = df['comment_text'].apply(clean_text)

print(df.shape)
df.info()

# --- STEP 2: Clean Duplicates ---
print("Deduplicating dataset...")

df = df.groupby('clean_text', as_index=False)[label_cols].max()

# Remove empty strings if cleaning resulted in blank comments
df = df[df['clean_text'].str.len() > 0].reset_index(drop=True)
print(f"Final Clean Dataset Shape: {df.shape}")

# --- STEP 3: Train / Validation Split ---
X_train, X_val, y_train, y_val = train_test_split(
    df['clean_text'].values,
    df[label_cols].values,
    test_size=0.15,
    random_state=42
)

print(f"Train size: {len(X_train):,}, Validation size: {len(X_val):,}")

Cleaning comment_text column...
(159571, 9)
<class 'pandas.DataFrame'>
RangeIndex: 159571 entries, 0 to 159570
Data columns (total 9 columns):
 #   Column         Non-Null Count   Dtype
---  ------         --------------   -----
 0   id             159571 non-null  str  
 1   comment_text   159571 non-null  str  
 2   toxic          159571 non-null  int64
 3   severe_toxic   159571 non-null  int64
 4   obscene        159571 non-null  int64
 5   threat         159571 non-null  int64
 6   insult         159571 non-null  int64
 7   identity_hate  159571 non-null  int64
 8   clean_text     159571 non-null  str  
dtypes: int64(6), str(3)
memory usage: 11.0 MB
Deduplicating dataset...
Final Clean Dataset Shape: (158647, 7)
Train size: 134,849, Validation size: 23,798


In [5]:
df.head()

,clean_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,! additionally the wta stuff follows the same ...,0,0,0,0,0,0
1,! and im not a sock puppet of gogo dodo but so...,0,0,0,0,0,0
2,! and no matter how hard you try to block me y...,0,0,0,0,0,0
3,! end group response above this line,0,0,0,0,0,0
4,! favorite cookie also having a little trouble...,0,0,0,0,0,0


## Vocab bulid + Tokens trancate

In [6]:
from collections import Counter


class Vocabulary:
    def __init__(self, max_size=10000):
        self.max_size = max_size
        self.pad_token = "<PAD>"  # Index 0
        self.unk_token = "<UNK>"  # Index 1
        
        self.word2idx = {self.pad_token: 0, self.unk_token: 1}
        self.idx2word = {0: self.pad_token, 1: self.unk_token}
        
    def tokenize(self, text):
        # Basic lowercase alphanumeric tokenization
        return re.findall(r'\b\w+\b', text.lower())

    def build_vocab(self, texts):
        word_counts = Counter()
        for text in texts:
            tokens = self.tokenize(text)
            word_counts.update(tokens)
            
        # Select most common words up to max_size - 2 (reserving 0 and 1)
        most_common = word_counts.most_common(self.max_size - 2)
        for word, _ in most_common:
            idx = len(self.word2idx)
            self.word2idx[word] = idx
            self.idx2word[idx] = word

    def encode(self, text, max_len=35):
        tokens = self.tokenize(text)
        # Numericalize
        indices = [self.word2idx.get(w, self.word2idx[self.unk_token]) for w in tokens]
        
        # Truncate or Pad to fixed sequence length (max_len)
        if len(indices) < max_len:
            indices = indices + [self.word2idx[self.pad_token]] * (max_len - len(indices))
        else:
            indices = indices[:max_len]
            
        return indices

# 2. Build Vocabulary using ONLY training data
vocab = Vocabulary(max_size=25000)
vocab.build_vocab(X_train)

print(f"Vocabulary built! Total unique tokens mapped: {len(vocab.word2idx):,}")


class MultiLabelTextDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len=150):
        self.texts = texts
        self.labels = labels
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        # Vocab encode automatically handles Truncation (if > max_len) and Padding (if < max_len)
        encoded = self.vocab.encode(text, max_len=self.max_len)
        
        return (
            torch.tensor(encoded, dtype=torch.long),
            torch.tensor(label, dtype=torch.float32)  
        )

Vocabulary built! Total unique tokens mapped: 25,000


In [7]:
# print tensors content 

train_dataset = MultiLabelTextDataset(X_train, y_train, vocab, max_len=150)
for i in range(6, 10):
    text_tensor, label_tensor = train_dataset[i]
    print(f"Text Tensor {i}: {text_tensor}")
    print(f"Label Tensor {i}: {label_tensor}")
    print(f"Shape of Text Tensor {i}: {text_tensor.size()}")
    print(f"Shape of Label Tensor {i}: {label_tensor.size()}")


Text Tensor 6: tensor([ 181,  132,    5,   15,  690,  107,   19,   77,   13, 1235,   27,    1,
          28,    7,  267,   61,   13,  274,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,

## Bulding The NN

In [8]:
import torch
import torch.nn as nn

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Model Hyperparameters
VOCAB_SIZE = len(vocab.word2idx)  # 25,000
EMBEDDING_DIM = 250             
HIDDEN_DIM = 128
NUM_CLASSES = 6                   

class BiLSTMToxicityModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim=250, hidden_dim=128, num_classes=6, dropout=0.3):
        super(BiLSTMToxicityModel, self).__init__()
        
        # Randomly initialized embedding layer (learns weights entirely from scratch)
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        
        # BiLSTM Layer
        self.lstm = nn.LSTM(
            embedding_dim, 
            hidden_dim, 
            num_layers=2, 
            batch_first=True, 
            bidirectional=True, 
            dropout=dropout
        )
        
        # Classification Head
        self.fc1 = nn.Linear(hidden_dim * 2, 64)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(64, num_classes)

    def forward(self, x):
        # x shape: [batch_size, max_len]
        embedded = self.embedding(x)                 # [batch_size, max_len, embedding_dim]
        lstm_out, _ = self.lstm(embedded)             # [batch_size, max_len, hidden_dim * 2]
        
        # Global Max Pooling across sequence length
        pooled, _ = torch.max(lstm_out, dim=1)        # [batch_size, hidden_dim * 2]
        
        out = self.fc1(pooled)
        out = self.relu(out)
        out = self.dropout(out)
        logits = self.fc2(out)                       # Raw logits for BCEWithLogitsLoss
        
        return logits

# Instantiate Model from scratch
model = BiLSTMToxicityModel(
    vocab_size=VOCAB_SIZE, 
    embedding_dim=EMBEDDING_DIM, 
    hidden_dim=HIDDEN_DIM, 
    num_classes=NUM_CLASSES
).to(device)

print(f"Model initialized with random embeddings! Total Trainable Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

Model initialized with random embeddings! Total Trainable Parameters: 7,051,222


### first we need to extract testing data from test CSVs

In [9]:
test_data = pd.read_csv('test.csv')
test_labels = pd.read_csv('test_labels.csv')

test_labels.head()


,id,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,00001cee341fdb12,-1,-1,-1,-1,-1,-1
1,0000247867823ef7,-1,-1,-1,-1,-1,-1
2,00013b17ad220c46,-1,-1,-1,-1,-1,-1
3,00017563c3f7919a,-1,-1,-1,-1,-1,-1
4,00017695ad8997eb,-1,-1,-1,-1,-1,-1


In [10]:
# create X_test and y_test from test_cleaned csv saved from test_data_check.ipynb

label_cols = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

df_test_clean = pd.read_csv('test_cleaned.csv')

X_test = df_test_clean['clean_text'].fillna('').astype(str).values
y_test = df_test_clean[label_cols].values


In [11]:
# Force exact label column order
label_cols = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

print("y_train shape:", y_train.shape)
print("y_test shape :", y_test.shape)

# Ensure y_test extracted strictly the 6 binary label columns in order
y_test = df_test_clean[label_cols].values

y_train shape: (134849, 6)
y_test shape : (63978, 6)


In [12]:
import pandas as pd
import numpy as np

label_cols = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

print("=== Positive Label Counts Comparison ===")
train_pos = y_train.sum(axis=0)
test_pos = y_test.sum(axis=0)

for i, col in enumerate(label_cols):
    train_pct = (train_pos[i] / len(y_train)) * 100
    test_pct = (test_pos[i] / len(y_test)) * 100
    print(f"{col:15s} | Train Pos: {train_pos[i]:6d} ({train_pct:5.2f}%) | Test Pos: {test_pos[i]:5d} ({test_pct:5.2f}%)")

=== Positive Label Counts Comparison ===
toxic           | Train Pos:  12937 ( 9.59%) | Test Pos:  6090 ( 9.52%)
severe_toxic    | Train Pos:   1348 ( 1.00%) | Test Pos:   367 ( 0.57%)
obscene         | Train Pos:   7133 ( 5.29%) | Test Pos:  3691 ( 5.77%)
threat          | Train Pos:    393 ( 0.29%) | Test Pos:   211 ( 0.33%)
insult          | Train Pos:   6681 ( 4.95%) | Test Pos:  3427 ( 5.36%)
identity_hate   | Train Pos:   1184 ( 0.88%) | Test Pos:   712 ( 1.11%)


In [13]:
# Inspect token encoding on a test sentence
test_sample_str = X_test[0]
encoded_indices = vocab.encode(test_sample_str, max_len=20)
decoded_tokens = [vocab.idx2word.get(idx, "<UNK>") for idx in encoded_indices if idx != 0]

print("Original text snippet :", test_sample_str[:80])
print("Encoded Tensor Indices:", encoded_indices[:10])
print("Decoded Tokens        :", " ".join(decoded_tokens[:10]))

Original text snippet : thank you for understanding i think very highly of you and would not revert with
Encoded Tensor Indices: [126, 8, 14, 982, 7, 68, 100, 1004, 5, 8]
Decoded Tokens        : thank you for understanding i think very highly of you


## Loss calc

In [14]:
from torch.utils.data import DataLoader
import torch.optim as optim
import numpy as np

# 1. Create DataLoaders
BATCH_SIZE = 64

train_dataset = MultiLabelTextDataset(X_train, y_train, vocab, max_len=150)
val_dataset = MultiLabelTextDataset(X_val, y_val, vocab, max_len=150)
test_dataset = MultiLabelTextDataset(X_test, y_test, vocab, max_len=150)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# 2. Compute Smoothed pos_weights for Class Imbalance
pos_counts = y_train.sum(axis=0)
total_train_samples = len(y_train)
neg_counts = total_train_samples - pos_counts

raw_pos_weights = neg_counts / pos_counts
smoothed_pos_weights = np.sqrt(raw_pos_weights)

pos_weight_tensor = torch.tensor(smoothed_pos_weights, dtype=torch.float32).to(device)

# 3. Loss & Optimizer
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

print("DataLoaders, Criterion, and Optimizer successfully initialized.")

DataLoaders, Criterion, and Optimizer successfully initialized.


# Training 

In [15]:
import time
from sklearn.metrics import roc_auc_score, f1_score

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    
    for texts, labels in loader:
        texts, labels = texts.to(device), labels.to(device)
        
        optimizer.zero_grad()
        logits = model(texts)
        loss = criterion(logits, labels)
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
    return running_loss / len(loader)

def evaluate(model, loader, criterion, device, threshold=0.5):
    model.eval()
    running_loss = 0.0
    all_targets = []
    all_preds = []
    
    with torch.no_grad():
        for texts, labels in loader:
            texts, labels = texts.to(device), labels.to(device)
            
            logits = model(texts)
            loss = criterion(logits, labels)
            running_loss += loss.item()
            
            # Apply Sigmoid to convert logits into probabilities [0, 1]
            probs = torch.sigmoid(logits)
            
            all_targets.append(labels.cpu().numpy())
            all_preds.append(probs.cpu().numpy())
            
    all_targets = np.vstack(all_targets)
    all_preds = np.vstack(all_preds)
    
    # Calculate Macro ROC-AUC across all 6 target classes
    macro_auc = roc_auc_score(all_targets, all_preds, average='macro')
    
    # Calculate Macro F1 score based on decision threshold
    binary_preds = (all_preds >= threshold).astype(int)
    macro_f1 = f1_score(all_targets, binary_preds, average='macro', zero_division=0)
    
    return running_loss / len(loader), macro_auc, macro_f1, all_targets, all_preds

In [16]:
import torch.optim as optim

# 1. Instantiate Model fresh
model = BiLSTMToxicityModel(
    vocab_size=VOCAB_SIZE, 
    embedding_dim=EMBEDDING_DIM, 
    hidden_dim=HIDDEN_DIM, 
    num_classes=NUM_CLASSES
).to(device)

# 2. Instantiate Loss Function (Unweighted)
criterion = nn.BCEWithLogitsLoss()

# 3. Instantiate Optimizer LINKED TO THIS SPECIFIC MODEL INSTANCE
optimizer = optim.Adam(model.parameters(), lr=1e-3)

print(f"Model, Criterion, and Optimizer synchronized on {device}!")

Model, Criterion, and Optimizer synchronized on cpu!


In [17]:
EPOCHS = 1
best_val_auc = 0.0

print("Starting Training Loop...")
print("=" * 70)

for epoch in range(EPOCHS):
    start_time = time.time()
    
    # 1. Train Epoch
    train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
    
    # 2. Validate Epoch
    val_loss, val_auc, val_f1, _, _ = evaluate(model, val_loader, criterion, device)
    
    elapsed = time.time() - start_time
    
    print(f"Epoch {epoch+1:02d}/{EPOCHS:02d} [{elapsed:.1f}s] | "
          f"Train Loss: {train_loss:.4f} | "
          f"Val Loss: {val_loss:.4f} | "
          f"Val ROC-AUC: {val_auc:.4f} | "
          f"Val Macro F1: {val_f1:.4f}")
    
    # 3. Model Checkpointing
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        torch.save(model.state_dict(), 'best_bilstm_no_glove.pth')
        print(f"  --> Saved new best checkpoint! (Val ROC-AUC: {best_val_auc:.4f})")

print("=" * 70)
print("Training Complete!")

Starting Training Loop...
Epoch 01/01 [851.3s] | Train Loss: 0.0789 | Val Loss: 0.0538 | Val ROC-AUC: 0.9680 | Val Macro F1: 0.3822
  --> Saved new best checkpoint! (Val ROC-AUC: 0.9680)
Training Complete!


## Evaluation

In [18]:
# Load the best checkpoint saved during training
model.load_state_dict(torch.load('best_bilstm_no_glove.pth', map_location=device))

# Run evaluation on test_loader
test_loss, test_auc, test_f1, test_targets, test_preds = evaluate(model, test_loader, criterion, device)

print("==================================================")
print("          HOLDOUT TEST SET EVALUATION             ")
print("==================================================")
print(f"Test Loss      : {test_loss:.4f}")
print(f"Test ROC-AUC   : {test_auc:.4f}")
print(f"Test Macro F1  : {test_f1:.4f}")
print("==================================================")

# Compute per-class ROC-AUC breakdown
label_cols = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
per_class_auc = roc_auc_score(test_targets, test_preds, average=None)

print("\n--- Per-Class ROC-AUC Breakdown ---")
for col, score in zip(label_cols, per_class_auc):
    print(f" - {col:15s}: {score:.4f}")

          HOLDOUT TEST SET EVALUATION             
Test Loss      : 0.0831
Test ROC-AUC   : 0.9558
Test Macro F1  : 0.3205

--- Per-Class ROC-AUC Breakdown ---
 - toxic          : 0.9538
 - severe_toxic   : 0.9809
 - obscene        : 0.9702
 - threat         : 0.9326
 - insult         : 0.9614
 - identity_hate  : 0.9360


In [19]:
import torch

model.eval()
with torch.no_grad():
    # Take 1 batch from test_loader
    texts, labels = next(iter(test_loader))
    texts, labels = texts.to(device), labels.to(device)
    
    # Get raw outputs from model
    raw_logits = model(texts)
    # Convert logits to probabilities (0.0 to 1.0)
    probs = torch.sigmoid(raw_logits)

print("=== Raw Model Outputs vs. Ground Truth ===")
print("1. Raw Logits (First 3 test comments):\n", raw_logits[:3].cpu().numpy().round(2))
print("\n2. Probabilities (0 to 1):\n", probs[:3].cpu().numpy().round(2))
print("\n3. Actual Correct Labels:\n", labels[:3].cpu().numpy().astype(int))

=== Raw Model Outputs vs. Ground Truth ===
1. Raw Logits (First 3 test comments):
 [[ -5.29 -13.67  -8.4   -9.89  -8.2   -9.48]
 [ -0.4   -4.86  -2.3   -3.65  -1.94  -3.52]
 [ -0.91  -6.4   -2.91  -4.83  -2.92  -4.73]]

2. Probabilities (0 to 1):
 [[0.01 0.   0.   0.   0.   0.  ]
 [0.4  0.01 0.09 0.03 0.13 0.03]
 [0.29 0.   0.05 0.01 0.05 0.01]]

3. Actual Correct Labels:
 [[0 0 0 0 0 0]
 [0 0 0 0 0 0]
 [0 0 0 0 0 0]]
